In [1]:
import requests

url = "https://www.traillink.com/trailsearch/?state=UT"

headers = {
    "accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
    "accept-encoding": "gzip, deflate, br, zstd",
    "accept-language": "en-US,en;q=0.9,id;q=0.8,ms;q=0.7",
    "cache-control": "max-age=0",
    "cookie": "G_ENABLED_IDPS=google; ASP.NET_SessionId=wu3apdcuedgufn4wwhug4mef",
    "priority": "u=0, i",
    "referer": "https://www.traillink.com/viewnationalmap/",
    "sec-ch-ua": "\"Microsoft Edge\";v=\"149\", \"Chromium\";v=\"149\", \"Not)A;Brand\";v=\"24\"",
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": "\"Windows\"",
    "sec-fetch-dest": "document",
    "sec-fetch-mode": "navigate",
    "sec-fetch-site": "same-origin",
    "sec-fetch-user": "?1",
    "upgrade-insecure-requests": "1",
    "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36 Edg/149.0.0.0"
}

try:
    response = requests.get(url, headers=headers)
    response.raise_for_status()  # Raise an exception for HTTP errors (4xx or 5xx)
    print(f"Status Code: {response.status_code}")
    print(f"Response Content (first 500 chars):\n{response.text[:500]}")
except requests.exceptions.RequestException as e:
    print(f"An error occurred: {e}")

Status Code: 200
Response Content (first 500 chars):
    <!DOCTYPE HTML>
    <html lang="" class="no-js">
    <head>
        <meta charset="utf-8">
        <meta name="viewport" content="width=device-width, initial-scale=1">
        <meta name="apple-itunes-app" content="app-id=636875425">
        <meta name="robots" content="max-image-preview:large">
                <meta name="description" content="Looking for a trail near you? Find the best hiking, biking, and walking trails in your area. Filter by activity, surface, or length. Search Tr


In [4]:
from bs4 import BeautifulSoup
import pandas as pd
import re

# Assuming 'response' is available from the previous cell's execution
# If you run this cell independently, ensure 'response' is defined, e.g:
# response = requests.get(url, headers=headers)

soup = BeautifulSoup(response.text, 'html.parser')

trails_data = []

# Find all trail entries
trail_divs = soup.find_all('div', class_='row collapse trail')

for trail in trail_divs:
    name = trail.find('span', itemprop='name').text.strip() if trail.find('span', itemprop='name') else 'N/A'
    distance = trail.find('span', itemprop='distance').text.strip() if trail.find('span', itemprop='distance') else 'N/A'
    state = trail.find('span', itemprop='addressRegion').text.strip() if trail.find('span', itemprop='addressRegion') else 'N/A'

    # Extract Trail URL
    trail_url_tag = trail.find('a', itemprop='url')
    trail_url = "https://www.traillink.com" + trail_url_tag['href'] if trail_url_tag and 'href' in trail_url_tag.attrs else 'N/A'

    # Extract Trail Image URL
    image_div = trail.find('div', class_='small-5 column image')
    image_url = 'N/A'
    if image_div and 'style' in image_div.attrs:
        style_attr = image_div['style']
        match = re.search(r"url\('([^']*)'\)", style_attr)
        if match:
            image_url = match.group(1)

    # Extract Surface
    surface = 'N/A'
    sub_div = trail.find('div', class_='sub')
    if sub_div:
        br_tags = sub_div.find_all('br')
        if len(br_tags) >= 2:
            surface_element = br_tags[1].find_next_sibling()
            surface = surface_element.text.strip() if surface_element else 'N/A'

    # Extract Total Rating (count stars)
    rating_div = trail.find('div', class_='stars')
    total_rating = len(rating_div.find_all('i', class_='fa-star')) if rating_div else 0

    trails_data.append({
        'Name': name,
        'Distance': distance,
        'State': state,
        'Surface': surface,
        'Trail_URL': trail_url,
        'Image_URL': image_url,
        'Total_Stars': total_rating
    })

# Create a pandas DataFrame
df_trails = pd.DataFrame(trails_data)
display(df_trails)

,Name,Distance,State,Surface,Trail_URL,Image_URL,Total_Stars
0,21st Street Pond Trail,1 mi,UT,Asphalt,https://www.traillink.com/trail/21st-street-po...,https://cloudfront.traillink.com/photos/21st-s...,5
1,Bryce Canyon Shared Use Path,5 mi,UT,Asphalt,https://www.traillink.com/trail/bryce-canyon-s...,https://cloudfront.traillink.com/photos/bryce-...,5
2,Candy Mountain Express Bike Trail,15.8 mi,UT,Asphalt,https://www.traillink.com/trail/candy-mountain...,https://cloudfront.traillink.com/photos/candy-...,5
3,Coal Creek Trail (UT),3.4 mi,UT,"Asphalt, Concrete",https://www.traillink.com/trail/coal-creek-tra...,https://cloudfront.traillink.com/photos/coal-c...,5
4,Denver and Rio Grande Western Rail Trail,23.5 mi,UT,Asphalt,https://www.traillink.com/trail/denver-and-rio...,https://cloudfront.traillink.com/photos/denver...,5
...,...,...,...,...,...,...,...
65,Point of the Mountain Trail,2.4 mi,UT,Asphalt,https://www.traillink.com/trail/point-of-the-m...,https://cloudfront.traillink.com/photos/point-...,0
66,Prospector Rail Trail,3 mi,UT,Asphalt,https://www.traillink.com/trail/prospector-rai...,https://cloudfront.traillink.com/photos/prospe...,0
67,Silver Quinns Trail,3.4 mi,UT,Asphalt,https://www.traillink.com/trail/silver-quinns-...,https://cloudfront.traillink.com/photos/silver...,0
68,Slick Rock Park Trail,0.3 mi,UT,Asphalt,https://www.traillink.com/trail/slick-rock-par...,https://cloudfront.traillink.com/photos/slick-...,0


In [5]:
import requests

new_url = "https://www.traillink.com/trail/bryce-canyon-shared-use-path/"

# Reuse the headers defined previously
try:
    response_single_trail = requests.get(new_url, headers=headers)
    response_single_trail.raise_for_status()  # Raise an exception for HTTP errors (4xx or 5xx)
    print(f"Status Code for single trail: {response_single_trail.status_code}")
    print(f"Response Content (first 500 chars) for single trail:\n{response_single_trail.text[:500]}")
except requests.exceptions.RequestException as e:
    print(f"An error occurred while fetching the single trail page: {e}")

Status Code for single trail: 200
Response Content (first 500 chars) for single trail:
    <!DOCTYPE HTML>
    <html lang="" class="no-js">
    <head>
        <meta charset="utf-8">
        <meta name="viewport" content="width=device-width, initial-scale=1">
        <meta name="apple-itunes-app" content="app-id=636875425">
        <meta name="robots" content="max-image-preview:large">
        
        
    
    
    <title>Bryce Canyon Shared Use Path | Utah Trails | TrailLink</title>
    <meta name="description" content="Bryce Canyon Shared Use Path spans 5 from Inspi


In [6]:
from bs4 import BeautifulSoup
import json
import re


def clean_text(text):
    if not text:
        return None
    return re.sub(r"\s+", " ", text).strip()


def extract_trail_info(html):
    soup = BeautifulSoup(html, "html.parser")

    data = {}

    # -------------------------------------------------------
    # Basic Info
    # -------------------------------------------------------
    title = soup.find("h1")
    data["name"] = clean_text(title.get_text()) if title else None

    state = soup.find("h3")
    data["state"] = clean_text(state.get_text()) if state else None

    canonical = soup.find("link", rel="canonical")
    data["url"] = canonical["href"] if canonical else None

    meta_desc = soup.find("meta", attrs={"name": "description"})
    data["summary"] = meta_desc.get("content") if meta_desc else None

    hero = soup.find("meta", itemprop="image")
    data["hero_image"] = hero.get("content") if hero else None

    # -------------------------------------------------------
    # Rating
    # -------------------------------------------------------
    rating = soup.find(attrs={"itemprop": "aggregateRating"})
    if rating:
        data["rating"] = {
            "score": rating.find("meta", itemprop="ratingValue")["content"],
            "review_count": rating.find("meta", itemprop="reviewCount")["content"],
        }

    # -------------------------------------------------------
    # Facts
    # -------------------------------------------------------
    facts = {}

    facts_section = soup.select("div.facts div")

    for item in facts_section:
        strong = item.find("strong")
        if not strong:
            continue

        key = clean_text(strong.get_text()).replace(":", "").lower().replace(" ", "_")

        if key == "activities":
            facts[key] = [
                clean_text(a.get_text())
                for a in item.select("a")
            ]
        else:
            value = item.find("span")
            facts[key] = clean_text(value.get_text()) if value else None

    data["facts"] = facts

    # -------------------------------------------------------
    # Coordinates
    # -------------------------------------------------------
    geo = soup.find(attrs={"itemprop": "geo"})
    if geo:
        lat = geo.find("meta", itemprop="latitude")
        lng = geo.find("meta", itemprop="longitude")

        data["coordinates"] = {
            "latitude": lat["content"] if lat else None,
            "longitude": lng["content"] if lng else None,
        }

    # -------------------------------------------------------
    # Description
    # -------------------------------------------------------
    desc = soup.find(attrs={"itemprop": "description"})

    if desc:
        paragraphs = []

        for p in desc.find_all("p", recursive=False):
            txt = clean_text(p.get_text(" ", strip=True))
            if txt:
                paragraphs.append(txt)

        data["description"] = "\n\n".join(paragraphs)

    # -------------------------------------------------------
    # Parking
    # -------------------------------------------------------
    parking = None

    parking_header = soup.find(id="trail-detail-parking")
    if parking_header:
        next_p = parking_header.find_next("p")
        if next_p:
            parking = clean_text(next_p.get_text(" ", strip=True))

    data["parking"] = parking

    # -------------------------------------------------------
    # Photos
    # -------------------------------------------------------
    photos = []

    for img in soup.select(".trail-photo img"):
        src = img.get("src")
        if src and src not in photos:
            photos.append(src)

    data["photos"] = photos

    # -------------------------------------------------------
    # Reviews
    # -------------------------------------------------------
    reviews = []

    for review in soup.select(".review"):
        item = {}

        title = review.find("h3", itemprop="name")
        body = review.find(attrs={"itemprop": "reviewBody"})
        author = review.find(attrs={"itemprop": "author"})
        date = review.find(attrs={"itemprop": "datePublished"})
        rating = review.find(attrs={"itemprop": "reviewRating"})

        if title:
            item["title"] = clean_text(title.get_text())

        if body:
            item["review"] = clean_text(body.get_text())

        if author:
            name = author.find("meta", itemprop="name")
            if name:
                item["author"] = name["content"]

        if date:
            item["date"] = date.get("datetime")

        if rating:
            score = rating.find("meta", itemprop="ratingValue")
            if score:
                item["rating"] = score["content"]

        reviews.append(item)

    data["reviews"] = reviews

    # -------------------------------------------------------
    # Nearby Trails
    # -------------------------------------------------------
    nearby = []

    for trail in soup.select(".nearby-trails section header"):
        a = trail.find("a")
        title = trail.find(class_="title")
        miles = trail.find(class_="miles")
        reviews = trail.find(class_="reviews")
        img = trail.find("img")

        nearby.append({
            "name": clean_text(title.get_text()) if title else None,
            "url": a["href"] if a else None,
            "image": img["src"] if img else None,
            "length": clean_text(miles.get_text().replace("Length:", "")) if miles else None,
            "reviews": clean_text(reviews.get_text()) if reviews else None,
        })

    data["nearby_trails"] = nearby

    return data


# -------------------------------------------------------
# Example
# -------------------------------------------------------
if __name__ == "__main__":

    result = extract_trail_info(response_single_trail.text)

    print(json.dumps(result, indent=2, ensure_ascii=False))

{
  "name": "Bryce Canyon Shared Use Path",
  "state": "Utah",
  "url": "https://www.traillink.com/trail/bryce-canyon-shared-use-path/",
  "summary": "Bryce Canyon Shared Use Path spans 5 from Inspiration Point in Bryce Canyon National Park to Bryce Canyon City . View amenities, descriptions, reviews, photos, itineraries, and directions on TrailLink.",
  "hero_image": "https://cloudfront.traillink.com/photos/bryce-canyon-shared-use-path_203727_hero.jpg",
  "rating": {
    "score": "5",
    "review_count": "4"
  },
  "facts": {
    "states": "Utah",
    "counties": "Garfield",
    "length": "5 miles",
    "trail_end_points": "Inspiration Point in Bryce Canyon National Park and Bryce Canyon City",
    "trail_surfaces": "Asphalt",
    "trail_category": "Greenway/Non-RT",
    "id": "10156772",
    "activities": [
      "Bike",
      "Inline Skating",
      "Walking",
      "Wheelchair Accessible",
      "Cross Country Skiing"
    ]
  },
  "coordinates": {
    "latitude": "37.670639924",
  